# 12 · Bring your own model

You have a PyTorch model. This notebook takes it from `nn.Module` to a wired-in
Brain-Score candidate in four steps — **inspect** it (auto-detect the wrapper and a
provisional region→layer map), **wrap** it, **extract** features at a mapped layer,
and print a **registration scaffold** you can drop into `brainscore/models/`.

This notebook uses an architecture with *random* weights (no download) and a few
synthetic images. Swap in your own module and stimuli.

## 1 · Your model

Any `nn.Module`. Here, a ResNet-18 with random weights.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import torchvision.models as tvm

# Any PyTorch model works. This one has random weights, so it downloads nothing and
# predicts nothing useful -- fine, because this notebook is about the WIRING, not scores.
model = tvm.resnet18(weights=None).eval()   # <-- your model goes here
n_params = sum(p.numel() for p in model.parameters())   # numel = number of values in a tensor
print(f'{model.__class__.__name__}: {n_params:,} parameters')

ResNet: 11,689,512 parameters


## 2 · Inspect it

`inspect_model` reads the architecture and recommends a wrapper plus a *provisional*
region→layer map — a starting point, not a committed mapping (you refine that with the
layer-mapping explorer against a real benchmark).

The region->layer map is a **structural heuristic** (it decides which layer UMI extracts when you request V1/V4/IT), *not* evidence of brain correspondence — you confirm the mapping empirically with the layer-mapping explorer.

In [2]:
from brainscore.tools.auto_register import inspect_model

# inspect_model reads the model's structure and suggests two things: which wrapper to
# use, and a first guess at which layer stands in for which brain region.
profile = inspect_model(model, identifier='my-resnet18')
print(profile.summary())
print('\nprovisional region_layer_map:')
# "Provisional" is the important word -- this is a starting point, not a finding. You
# confirm it later by scoring candidate layers against real data.
for region, layer in profile.provisional_region_layer_map().items():
    print(f'  {region:3s} -> {layer}')

Model: my-resnet18
  parameters: 11,689,512  dtype: float32
  multimodal: False
  tower 0: vision  → VisionWrapper
    wrap: (full model)
    8 candidate layers (layer1.0 … layer4.1)
    why: vision backbone signal in 'ResNet'
    provisional map: V1=layer1.0, V2=layer2.0, V4=layer3.1, IT=layer4.1

provisional region_layer_map:
  V1  -> layer1.0
  V2  -> layer2.0
  V4  -> layer3.1
  IT  -> layer4.1


## 3 · Wrap it and extract features at a mapped layer

`inspect_model` recommended **`VisionWrapper`**, so that is what we use. It is the front
door for vision: it looks at your model and dispatches internally to `PytorchWrapper`
(plain image models), `VLMVisionWrapper` (VLMs, whose patches arrive concatenated rather
than stacked), or `VideoWrapper` (native-temporal models). One name covers all three, so
you do not have to categorise your model yourself.

You will see the concrete classes named directly in some notebooks — that is the same
machinery one level down, used where seeing the moving parts helps. For registering a
model, `VisionWrapper` is the shorter path.

We build a few synthetic images; point it at your own images the same way.

In [3]:
import tempfile, os, numpy as np
from PIL import Image
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore_vision.model_helpers.activations.pytorch import load_preprocess_images

# Four random 64x64 images, standing in for a real stimulus set.
d = tempfile.mkdtemp()
paths = []
for i in range(4):
    p = os.path.join(d, f'img{i}.png')
    Image.fromarray((np.random.RandomState(i).rand(64, 64, 3) * 255).astype('uint8')).save(p)
    paths.append(p)

# VisionWrapper is the one name to remember for vision: it inspects your model and picks
# the right concrete wrapper internally (here, PytorchWrapper).
it_layer = profile.provisional_region_layer_map()['IT']
wrapper = VisionWrapper(model, identifier='my-resnet18',
                        preprocessing=lambda imgs: load_preprocess_images(imgs, image_size=64))

# Run the images through and keep the values from the IT layer.
activations = wrapper(paths, layers=[it_layer])
print(f'features at IT layer ({it_layer}): {activations.shape}  '
      f'({activations.sizes["presentation"]} images x {activations.sizes["neuroid"]} units)')

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

features at IT layer (layer4.1): (4, 2048)  (4 images x 2048 units)


## 4 · The registration scaffold

`scaffold_registration` emits a starting-point plugin. Fill in the real model loader +
preprocessing, refine `REGION_LAYER_MAP` with the layer-mapping explorer, drop it in
`brainscore/models/<name>/`, and score it through the
`load_model → load_benchmark → score` path.

The scaffold is a template: it ends in `NotImplementedError` until you fill in the loader. The working proof in this notebook is the feature extraction above — `(4, 2048)` real activations at the mapped layer.

In [4]:
from brainscore.tools.auto_register import scaffold_registration

# Print a starting-point registration file: the imports, the wrapper, and the region map,
# already filled in from what inspect_model found. Copy it into brainscore/models/<name>/
# and replace the model loader with your own.
print(scaffold_registration(profile))

"""Auto-generated registration scaffold for 'my-resnet18'.

Generated by brainscore.tools.auto_register. Fill in the model loader +
preprocessing, then refine REGION_LAYER_MAP with the layer-mapping explorer:

    from brainscore.tools.layer_mapping import sweep_model
    result, approaches, selector = sweep_model(
        wrapper, stimulus_set, brain_target, brain_stimulus_ids,
        layers=['layer1.0', 'layer1.1', 'layer2.0', 'layer2.1', 'layer3.0', 'layer3.1', 'layer4.0', 'layer4.1'])
    print(result.best_layer)   # → assign to the region you are mapping
"""
from brainscore import model_registry
from brainscore_core.model_interface import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

# Provisional map (evenly spaced over detected blocks — REFINE with the
# layer-mapping explorer; this is only a runnable starting point).
REGION_LAYER_MAP = {
    'V1': 'layer1.0',
    'V2': 'layer2.0',
    'V4': 'layer3.1',
    'IT': 'layer4.1',
}


def get_mode

**Next:** refine the mapping with `brainscore.tools.layer_mapping` against a real
benchmark, register the model, then `brainscore.score('my-resnet18', '<benchmark>')`.
See [EXTENDING.md](../EXTENDING.md).